# dataset mahasiswa

In [1]:
# ==============================================================================
# DATASET FLATTENING PER TASK FOR JPLAG
# ==============================================================================
import shutil
import re
from pathlib import Path
from tqdm.notebook import tqdm

def flatten_dataset_per_task(input_dir: str, output_dir: str, overwrite: bool = True):
    """
    Mengubah struktur direktori dari (NIM/MODUL/file.py) menjadi (MODUL__FILE/NIM.py)
    untuk mengisolasi pengujian plagiarisme berbasis penugasan spesifik pada JPlag.
    """
    print(f"🔄 Memulai flattening JPlag per task\n📁 Input : {input_dir}")
    input_dir, output_dir = Path(input_dir), Path(output_dir)

    if overwrite and output_dir.exists(): shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    total_files, total_groups = 0, set()

    for py_file in tqdm(list(input_dir.rglob("*.py")), desc="Flattening Dataset", unit="file"):
        relative = py_file.relative_to(input_dir)
        if len(relative.parts) < 3: continue

        # Identifikasi komponen struktur direktori sumber data
        nim, modul = relative.parts[0], relative.parts[1]
        filename = py_file.stem
        match = re.match(
            rf"{nim}_{modul}_(.+)",
            filename
        )
        if not match:
            print(
                f"⚠ Format tidak dikenali: {filename}"
            )
            continue
        task_name = match.group(1)

        group_name = f"{modul}__{task_name}"

        # Penyelarasan folder tujuan berdasarkan task penugasan
        target_dir = output_dir / group_name
        target_dir.mkdir(parents=True, exist_ok=True)
        
        shutil.copy2(py_file, target_dir / f"{nim}.py")
        total_groups.add(group_name)
        total_files += 1

    print("\n✅ Flattening selesai")
    print(f"📄 Total file : {total_files:,}")
    print(f"📂 Total grup : {len(total_groups):,}")
    print(f"📁 Output     : {output_dir}")
    
INPUT_DIR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\dataset(2)\06_Filtered_tgs"  # Folder asal dataset utama mahasiswa yang sudah difilter bersih total
OUTPUT_DIR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\input_jplag_tgs"  # Folder target untuk dataset yang sudah diflattening siap untuk JPlag

flatten_dataset_per_task(input_dir=INPUT_DIR, output_dir=OUTPUT_DIR)

🔄 Memulai flattening JPlag per task
📁 Input : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\dataset(2)\06_Filtered_tgs


Flattening Dataset:   0%|          | 0/566 [00:00<?, ?file/s]


✅ Flattening selesai
📄 Total file : 566
📂 Total grup : 12
📁 Output     : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\input_jplag_tgs


In [2]:
INPUT_DIR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\dataset(2)\06_Filtered_prak"  # Folder asal dataset utama mahasiswa yang sudah difilter bersih total
OUTPUT_DIR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\input_jplag_prak"  # Folder target untuk dataset yang sudah diflattening siap untuk JPlag

flatten_dataset_per_task(input_dir=INPUT_DIR, output_dir=OUTPUT_DIR)

🔄 Memulai flattening JPlag per task
📁 Input : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\dataset(2)\06_Filtered_prak


Flattening Dataset:   0%|          | 0/1678 [00:00<?, ?file/s]


✅ Flattening selesai
📄 Total file : 1,678
📂 Total grup : 35
📁 Output     : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\input_jplag_prak


In [5]:
# ==============================================================================
# RUN JPLAG PER TASK
# ==============================================================================
import subprocess, time
from pathlib import Path
import os, shutil
import pandas as pd

def run_jplag_per_task(input_root: str, output_root: str, dataset_type, jplag_jar: str, language: str = "python3"):
    """
    Menjalankan pengujian JPlag secara batch/sekuensial untuk setiap grup tugas (task).
    Menghasilkan berkas arsip report .jplag dan ringkasan dataset matriks .csv.
    """
    input_root, output_root, jplag_jar = Path(input_root), Path(output_root), Path(jplag_jar)
    
    if os.path.exists(output_root):
        print(f"⚠️ Output root '{output_root}' sudah ada, menghapus untuk eksekusi bersih...")
        shutil.rmtree(output_root)
    output_root.mkdir(parents=True, exist_ok=True)
    
    if not input_root.exists(): raise FileNotFoundError(input_root)
    if not jplag_jar.exists(): raise FileNotFoundError(jplag_jar)

    task_dirs = sorted([d for d in input_root.iterdir() if d.is_dir()])

    total_files_all = 0
    total_pairs_all = 0
    total_runtime = 0
    
    print("=" * 80)
    print(f"JPLAG BATCH EXECUTION {dataset_type}")
    print("=" * 80)
    print(f"Total Task : {len(task_dirs)}")
    print()
    execution_logs = []
    
    for idx, task_dir in enumerate(task_dirs, start=1):
        task_name = task_dir.name
        print(f"[{idx}/{len(task_dirs)}] Processing {task_name}")

        file_count = len(
            list(task_dir.glob("*.py"))
        )
        pair_count = (file_count * (file_count - 1)) // 2

        total_files_all += file_count
        total_pairs_all += pair_count

        print(f"    Files : {file_count:,}")

        task_output = output_root / task_name
        task_output.mkdir(parents=True, exist_ok=True)
        result_file = task_output / "results"

        # Parameter CLI JPlag untuk optimasi eksekusi batch
        cmd = [
            "java", "-jar", str(jplag_jar), "-l", language, 
            "--csv-export", "--cluster-skip", "--overwrite", 
            "-r", str(result_file), str(task_dir)
        ]

        start_time = time.time()
        process = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8")
        elapsed = time.time() - start_time
        total_runtime += elapsed
        # Flushing output biner stdout JPlag ke file log lokal per-task
        with open(task_output / "jplag.log", "w", encoding="utf-8") as f:
            f.write(process.stdout)

        status = "SUCCESS" if process.returncode == 0 else "FAILED"
        execution_logs.append({
            "task_name": task_name,
            "file_count": file_count,
            "pair_count": pair_count,
            "status": status,
            "runtime_seconds": round(elapsed, 2)
        })
        
        # Logging konsol terpisah untuk keterbacaan info status
        print(f"    Status : {status}")
        print(f"    Pairs  : {pair_count:,}")
        print(f"    Time   : {elapsed:.2f}s")
        print(f"    Output : {task_output}\n")

    # ==========================================================
    # EXPORT EXECUTION REPORT
    # ==========================================================
    report_path = output_root / "jplag_execution_report.xlsx"
    execution_df = pd.DataFrame(execution_logs)
    summary_df = pd.DataFrame([
        {
            "metric": "total_task",
            "value": len(task_dirs)
        },
        {
            "metric": "total_files",
            "value": total_files_all
        },
        {
            "metric": "total_pairs",
            "value": total_pairs_all
        },
        {
            "metric": "total_runtime_seconds",
            "value": round(total_runtime, 2)
        },
        {
            "metric": "average_runtime_seconds",
            "value": round(
                total_runtime / len(task_dirs),
                2
            ) if len(task_dirs) > 0 else 0
        }
    ])

    with pd.ExcelWriter(report_path,engine="openpyxl") as writer:
        execution_df.to_excel(
            writer,
            sheet_name="task_execution_log",
            index=False
        )
        summary_df.to_excel(
            writer,
            sheet_name="summary",
            index=False
        )

    print(f"\n📊 Execution report saved:")
    print(report_path)
    print("=" * 80)
    print(f"{'SEMUA TASK SELESAI':^80}")
    print(f"Total Files : {total_files_all:,}")
    print(f"Total Pairs : {total_pairs_all:,}")
    print(f"Total Time  : {total_runtime:.2f}s")
    print(
        f"Average Runtime : "
        f"{(total_runtime / len(task_dirs)):.2f} detik/task"
        if task_dirs else
        "Average Runtime : 0"
    )
    print("=" * 80)


# EKSEKUSI BATCH JPLAG PER TASK
JPLAG_JAR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\jplag-5.1.0.jar"
INPUT_ROOT = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\input_jplag_prak"
OUTPUT_ROOT = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_prak"

run_jplag_per_task(
    input_root=INPUT_ROOT,
    output_root=OUTPUT_ROOT,
    jplag_jar=JPLAG_JAR,
    dataset_type='Praktikum'
)

JPLAG BATCH EXECUTION Praktikum
Total Task : 35

[1/35] Processing js02__p01
    Files : 52
    Status : SUCCESS
    Pairs  : 1,326
    Time   : 14.16s
    Output : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_prak\js02__p01

[2/35] Processing js02__p02
    Files : 49
    Status : SUCCESS
    Pairs  : 1,176
    Time   : 10.96s
    Output : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_prak\js02__p02

[3/35] Processing js02__p03
    Files : 49
    Status : SUCCESS
    Pairs  : 1,176
    Time   : 10.23s
    Output : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_prak\js02__p03

[4/35] Processing js02__p04
    Files : 49
    Status : SUCCESS
    Pairs  : 1,176
    Time   : 16.47s
    Output : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_prak\js02__p04

[5/35] Processing js03__p01
    Files : 53
    Status : SUCCESS
    Pairs  : 1,378
    Time   : 22.24s
    Output : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_prak\js

In [6]:
# EKSEKUSI BATCH JPLAG PER TASK
JPLAG_JAR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\jplag-5.1.0.jar"
INPUT_ROOT = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\input_jplag_tgs"
OUTPUT_ROOT = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_tgs"

run_jplag_per_task(
    input_root=INPUT_ROOT,
    output_root=OUTPUT_ROOT,
    jplag_jar=JPLAG_JAR,
    dataset_type='TUGAS'
)

JPLAG BATCH EXECUTION TUGAS
Total Task : 12

[1/12] Processing js02__tp
    Files : 49
    Status : SUCCESS
    Pairs  : 1,176
    Time   : 11.67s
    Output : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_tgs\js02__tp

[2/12] Processing js03__tp
    Files : 49
    Status : SUCCESS
    Pairs  : 1,176
    Time   : 12.18s
    Output : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_tgs\js03__tp

[3/12] Processing js04__tp
    Files : 45
    Status : SUCCESS
    Pairs  : 990
    Time   : 19.08s
    Output : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_tgs\js04__tp

[4/12] Processing js05__tp
    Files : 48
    Status : SUCCESS
    Pairs  : 1,128
    Time   : 15.39s
    Output : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_tgs\js05__tp

[5/12] Processing js06__tp
    Files : 46
    Status : SUCCESS
    Pairs  : 1,035
    Time   : 11.25s
    Output : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_tgs\js06__tp

[6/12] Proce

In [7]:
# ==============================================================================
# GENERATE JPLAG EXCEL REPORT
# ==============================================================================
import sys

import pandas as pd
from pathlib import Path

def generate_jplag_excel_report(jplag_output_dir: str, excel_path: str, type_dir):
    """
    Menggabungkan seluruh hasil CSV JPlag dari pengerjaan tugas mahasiswa
    menjadi satu berkas laporan akhir Excel multi-sheet terpadu.
    """
    jplag_output_dir = Path(jplag_output_dir)
    all_rows, unique_nims, total_files = [], set(), 0

    # Penelusuran direktori output task biner JPlag
    for task_dir in sorted(jplag_output_dir.iterdir()):
        if not task_dir.is_dir(): continue
        csv_file = task_dir / "results/results.csv"

        if not csv_file.exists(): continue

        task_name = task_dir.name
        try: modul, file_name = task_name.split("__", 1)
        except ValueError: modul, file_name = task_name, "-"

        df = pd.read_csv(csv_file)
        submissions = set()

        for _, row in df.iterrows():
            nim1, nim2 = Path(str(row["submissionName1"])).stem, Path(str(row["submissionName2"])).stem
            submissions.update([nim1, nim2])
            unique_nims.update([nim1, nim2])

            # Pindai kecocokan skema nama kolom similarity dari log JPlag
            if "averageSimilarity" in df.columns: similarity = row["averageSimilarity"]
            elif "avgSimilarity" in df.columns: similarity = row["avgSimilarity"]
            elif "similarity" in df.columns: similarity = row["similarity"]
            else: similarity = 0

            all_rows.append({
                "modul": modul, "file": file_name, "nim1": nim1, "nim2": nim2,
                "similarity_score": round(similarity * 100, 2)
            })
        total_files += len(submissions)

    # Inisialisasi DataFrame Utama
    all_similarity_df = pd.DataFrame(all_rows)

    # Transformasi Agregasi Agregat : Hitung Rata-rata per-Modul (Groupby)
    modul_summary_df = all_similarity_df.groupby("modul", as_index=False).agg(
        average_similarity_score=("similarity_score", "mean")
    )
    modul_summary_df["average_similarity_score"] = modul_summary_df["average_similarity_score"].round(2)

    # Kompilasi Summary Statistik Makro Eksperimen Baseline
    total_pairs = len(all_similarity_df)
    overall_summary_df = pd.DataFrame({
        "metric": ["total_nim", "total_modul", "total_files", "total_pairs"],
        "value": [len(unique_nims), all_similarity_df["modul"].nunique(), total_files, total_pairs]
    })

    # Konversi data numerik menjadi representasi string persentase (%) laporan
    all_similarity_df["similarity_score"] = all_similarity_df["similarity_score"].map(lambda x: f"{x:.2f}%")
    modul_summary_df["average_similarity_score"] = modul_summary_df["average_similarity_score"].map(lambda x: f"{x:.2f}%")

    # Flush massal DataFrame kolektif kembali ke berkas Excel menggunakan context manager
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        all_similarity_df.to_excel(writer, sheet_name="all_similarity", index=False)
        modul_summary_df.to_excel(writer, sheet_name="modul_summary", index=False)
        overall_summary_df.to_excel(writer, sheet_name="overall_summary", index=False)

    print("=" * 70)
    print(f"JPLAG EXCEL REPORT GENERATED {type_dir}")
    print("=" * 70)
    print(f"Total NIM   : {len(unique_nims):,}")
    print(f"Total Modul : {all_similarity_df['modul'].nunique():,}")
    print(f"Total Files : {total_files:,}")
    print(f"Total Pairs : {total_pairs:,}")
    print(f"Output      : {excel_path}")
    print("=" * 70)

# EKSEKUSI GENERATE EXCEL REPORT JPLAG
OUTPUT_JPLAG_DIR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_prak"
EXCEL_REPORT_PATH = Path(OUTPUT_JPLAG_DIR) / "jplag_similarity_report.xlsx"
generate_jplag_excel_report(
    jplag_output_dir=OUTPUT_JPLAG_DIR,
    excel_path=EXCEL_REPORT_PATH,
    type_dir="PRAKTIKUM"
)

JPLAG EXCEL REPORT GENERATED PRAKTIKUM
Total NIM   : 53
Total Modul : 11
Total Files : 1,678
Total Pairs : 39,457
Output      : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_prak\jplag_similarity_report.xlsx


In [8]:
# EKSEKUSI GENERATE EXCEL REPORT JPLAG
OUTPUT_JPLAG_DIR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_tgs"
EXCEL_REPORT_PATH = Path(OUTPUT_JPLAG_DIR) / "jplag_similarity_report.xlsx"
generate_jplag_excel_report(
    jplag_output_dir=OUTPUT_JPLAG_DIR,
    excel_path=EXCEL_REPORT_PATH,
    type_dir="TUGAS"
)

JPLAG EXCEL REPORT GENERATED TUGAS
Total NIM   : 50
Total Modul : 11
Total Files : 566
Total Pairs : 13,087
Output      : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_tgs\jplag_similarity_report.xlsx


# dataset eval

In [1]:
# ==============================================================================
# FLATTEN EVALUATION DATASET FOR JPLAG
# ==============================================================================
import shutil
from pathlib import Path
from tqdm.notebook import tqdm
import time
from datetime import datetime

def flatten_eval_dataset(input_root: str, output_dir: str, type_dir, overwrite: bool = True):
    """
    Mengumpulkan seluruh file .py dari subfolder evaluasi ke dalam satu direktori datar.
    """
    input_root = Path(input_root)
    output_dir = Path(output_dir)

    if overwrite and output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Identifikasi file sumber
    source_folders = ["asli", "type1", "type2", "type3", "type4"]
    all_files = []
    for folder in source_folders:
        source_dir = input_root / folder
        if source_dir.exists():
            all_files.extend(list(source_dir.glob("*.py")))

    # Proses penyalinan
    for py_file in tqdm(all_files, desc="Flattening", unit="file"):
        shutil.copy2(py_file, output_dir / py_file.name)

    total_files = len(all_files)
    total_pairs = (total_files * (total_files - 1)) // 2

    print("=" * 80)
    print(f"{'FLATTEN EVALUATION DATASET {type_dir} ':^80}")
    print("=" * 80)
    print(f"Files : {total_files:,}")
    print(f"Pairs : {total_pairs:,}")
    print(f"Output: {output_dir}")
    print("=" * 80)

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ dijalankan pada {timestamp}")

✅ dijalankan pada 2026-06-17 03:19:52


In [2]:
INPUT_DIR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\dataset(2)_eval\praktikum\06_SAMPLE"
OUTPUT_DIR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\input_jplag_eval_prak"

flatten_eval_dataset(input_root=INPUT_DIR, output_dir=OUTPUT_DIR, type_dir="PRAKTIKUM")

Flattening:   0%|          | 0/2515 [00:00<?, ?file/s]

                     FLATTEN EVALUATION DATASET {type_dir}                      
Files : 2,515
Pairs : 3,161,355
Output: D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\input_jplag_eval_prak


In [3]:
INPUT_DIR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\dataset(2)_eval\tugas\06_SAMPLE"
OUTPUT_DIR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\input_jplag_eval_tgs"

flatten_eval_dataset(input_root=INPUT_DIR, output_dir=OUTPUT_DIR, type_dir='TUGAS')

Flattening:   0%|          | 0/850 [00:00<?, ?file/s]

                     FLATTEN EVALUATION DATASET {type_dir}                      
Files : 850
Pairs : 360,825
Output: D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\input_jplag_eval_tgs


In [4]:
# ==============================================================================
# JPLAG EVALUATION DATASET
# ==============================================================================
import subprocess, time
from pathlib import Path

def run_jplag_eval(input_dir: str, output_dir: str, jplag_jar: str, type_dir, language: str = "python3"):
    """
    Menjalankan pengujian engine baseline JPlag pada flat dataset evaluasi biner controls.
    Mengekspor matriks pasangan kecurangan buatan untuk kalkulasi metrik klasifikasi.
    """
    input_dir, output_dir, jplag_jar = Path(input_dir), Path(output_dir), Path(jplag_jar)

    if not input_dir.exists(): raise FileNotFoundError(f"Input directory tidak ditemukan: {input_dir}")
    if not jplag_jar.exists(): raise FileNotFoundError(f"JPlag jar tidak ditemukan: {jplag_jar}")
    output_dir.mkdir(parents=True, exist_ok=True)

    # Kalkulasi kuantitas berkas dan pasangannya secara matematikal kombinatorial
    total_files = len(list(input_dir.glob("*.py")))
    total_pairs = (total_files * (total_files - 1)) // 2
    result_file = output_dir / "results"

    # Inisialisasi daftar parameter tokenization biner CLI JPlag
    cmd = [
        "java", "-jar", str(jplag_jar), "-l", language,
        "--csv-export", "--cluster-skip", "--overwrite",
        "-r", str(result_file), str(input_dir)
    ]

    print("=" * 80)
    print(f"JPLAG EVALUATION EXECUTION {type_dir}")
    print("=" * 80)
    print(f"Input Directory : {input_dir}")
    print(f"Output Directory: {output_dir}")
    print(f"Total Files     : {total_files:,}")
    print(f"Total Pairs     : {total_pairs:,}")
    print("\nCommand:")
    print(" ".join(cmd))
    print("\nRunning JPlag ...\n")

    start_time = time.time()
    process = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8")
    elapsed = time.time() - start_time
    status = "SUCCESS" if process.returncode == 0 else "FAILED"
    # Simpan record dump runtime stdout JPlag ke file teks log
    with open(output_dir / "jplag.log", "w", encoding="utf-8") as f:
        f.write("=" * 80 + "\n")
        f.write("JPLAG EVALUATION LOG\n")
        f.write("=" * 80 + "\n")
        f.write(f"Input Directory : {input_dir}\n")
        f.write(f"Output Directory: {output_dir}\n")
        f.write(f"Total Files     : {total_files}\n")
        f.write(f"Total Pairs     : {total_pairs}\n")
        f.write(f"Command         : {' '.join(cmd)}\n")
        f.write("\n--- JPLAG OUTPUT ---\n\n")
        f.write(process.stdout)
        f.write("\n\n--- EXECUTION SUMMARY ---\n")
        f.write(f"Status          : {status}\n")
        f.write(f"Runtime         : {elapsed:.2f} detik\n")

    print("\n" + "=" * 80)
    print(f"Status        : {status}")
    print(f"Runtime       : {elapsed:.2f} detik")
    print(f"Log File      : {output_dir / 'jplag.log'}")
    print(f"Result Prefix : {result_file}")
    print("=" * 80)

    return process.returncode

In [5]:
# EKSEKUSI JPLAG EVALUATION
JPLAG_JAR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\jplag-5.1.0.jar"
INPUT_DIR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\input_jplag_eval_prak"
OUTPUT_DIR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_eval_prak"

run_jplag_eval(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    jplag_jar=JPLAG_JAR,
    type_dir = 'PRAKTIKUM'
)

JPLAG EVALUATION EXECUTION PRAKTIKUM
Input Directory : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\input_jplag_eval_prak
Output Directory: D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_eval_prak
Total Files     : 2,515
Total Pairs     : 3,161,355

Command:
java -jar D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\jplag-5.1.0.jar -l python3 --csv-export --cluster-skip --overwrite -r D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_eval_prak\results D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\input_jplag_eval_prak

Running JPlag ...


Status        : SUCCESS
Runtime       : 332.44 detik
Log File      : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_eval_prak\jplag.log
Result Prefix : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_eval_prak\results


0

In [6]:
# EKSEKUSI JPLAG EVALUATION
JPLAG_JAR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\jplag-5.1.0.jar"
INPUT_DIR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\input_jplag_eval_tgs"
OUTPUT_DIR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_eval_tgs"

run_jplag_eval(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    jplag_jar=JPLAG_JAR,
    type_dir = 'TUGAS'
)

JPLAG EVALUATION EXECUTION TUGAS
Input Directory : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\input_jplag_eval_tgs
Output Directory: D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_eval_tgs
Total Files     : 850
Total Pairs     : 360,825

Command:
java -jar D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\jplag-5.1.0.jar -l python3 --csv-export --cluster-skip --overwrite -r D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_eval_tgs\results D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\input_jplag_eval_tgs

Running JPlag ...


Status        : SUCCESS
Runtime       : 167.83 detik
Log File      : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_eval_tgs\jplag.log
Result Prefix : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_eval_tgs\results


0

In [7]:
# ==============================================================================
# EVALUASI JPLAG MENGGUNAKAN PAIRS YANG SAMA DENGAN GRAPH2VEC
# ==============================================================================
import pandas as pd
import os
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

def normalize_filename(filename):
    filename = os.path.basename(
        str(filename)
    )
    if filename.endswith(".py"):
        filename = filename[:-3]

    return filename.strip()

# ==============================================================================
# EVALUASI JPLAG MENGGUNAKAN PAIRS YANG SAMA DENGAN GRAPH2VEC (REFACTORED)
# ==============================================================================

def evaluate_jplag_from_pairs(evaluation_excel, jplag_csv, output_excel, threshold=80.0):
    pairs_df = pd.read_excel(evaluation_excel, sheet_name="pairs_prediction_log")
    jplag_df = pd.read_csv(jplag_csv)

    print(f"Pairs Evaluation : {len(pairs_df):,}")
    print(f"JPlag Results    : {len(jplag_df):,}")

    # ------------------------------------------------------------------
    # NORMALISASI & DETEKSI KOLOM
    # ------------------------------------------------------------------
    jplag_df["file_1"] = jplag_df["submissionName1"].apply(normalize_filename)
    jplag_df["file_2"] = jplag_df["submissionName2"].apply(normalize_filename)

    if "averageSimilarity" in jplag_df.columns:
        similarity_col = "averageSimilarity"
    elif "maxSimilarity" in jplag_df.columns:
        similarity_col = "maxSimilarity"
    else:
        raise ValueError("Kolom similarity tidak ditemukan pada CSV JPlag")

    jplag_df["similarity_score"] = jplag_df[similarity_col] * 100

    # ------------------------------------------------------------------
    # BIDIRECTIONAL LOOKUP & EVALUASI
    # ------------------------------------------------------------------
    pair_lookup = {}
    matched_pairs = 0
    missing_pairs = 0
    for _, row in jplag_df.iterrows():
        f1 = row["file_1"]
        f2 = row["file_2"]

        key = tuple(sorted([f1, f2]))

        pair_lookup[key] = row["similarity_score"]
    
    results = []
    for _, row in pairs_df.iterrows():
        file_1 = normalize_filename(row["file_1"])
        file_2 = normalize_filename(row["file_2"])

        lookup_key = tuple(sorted([file_1, file_2]))

        if lookup_key in pair_lookup:
            similarity = pair_lookup[lookup_key]
            matched_pairs += 1
        else:
            similarity = 0.0
            missing_pairs += 1
            
        prediction = 1 if similarity >= threshold else 0
        
        results.append({
            "clone_type": row.get("clone_type", "unknown"),
            "file_1": file_1,
            "file_2": file_2,
            "jplag_similarity_score": round(similarity, 2),
            "ground_truth": row["ground_truth"],
            "prediction": prediction,
            "result": "CORRECT" if prediction == row["ground_truth"] else "WRONG"
        })

    result_df = pd.DataFrame(results)
    y_true, y_pred = result_df["ground_truth"].astype(int), result_df["prediction"].astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    roc_auc = roc_auc_score(
        y_true,
        result_df["jplag_similarity_score"]
    )

    # ------------------------------------------------------------------
    # METRIK & SUMMARY
    # ------------------------------------------------------------------
    metrics_df = pd.DataFrame([{
        "Threshold (%)": threshold,
        "Accuracy (%)": round(accuracy_score(y_true, y_pred) * 100, 2),
        "Precision (%)": round(precision_score(y_true, y_pred, zero_division=0) * 100, 2),
        "Recall (%)": round(recall_score(y_true, y_pred, zero_division=0) * 100, 2),
        "F1 Score (%)": round(f1_score(y_true, y_pred, zero_division=0) * 100, 2),
        "True Positive": tp, "False Positive": fp, "False Negative": fn, "True Negative": tn,
        "ROC-AUC"   : roc_auc,
        "Total Data": len(result_df)
    }])

    clone_summary = result_df.groupby("clone_type").agg(
        total_pairs=("clone_type", "count"),
        avg_similarity=("jplag_similarity_score", "mean"),
        min_similarity=("jplag_similarity_score", "min"),
        max_similarity=("jplag_similarity_score", "max"),
        positive_prediction=("prediction", "sum")
    ).reset_index()
    clone_summary["avg_similarity"] = clone_summary["avg_similarity"].round(2)
    clone_summary["detection_rate"] = (
        clone_summary["positive_prediction"]
        / clone_summary["total_pairs"] * 100
    ).round(2)

    # ------------------------------------------------------------------
    # DETAIL METRICS PER CLONE TYPE
    # ------------------------------------------------------------------
    type_metrics = []
    for clone_type, group in result_df.groupby("clone_type"):
        y_true_type = group["ground_truth"].astype(int)
        y_pred_type = group["prediction"].astype(int)
        tn_t, fp_t, fn_t, tp_t = confusion_matrix(
            y_true_type,
            y_pred_type,
            labels=[0, 1]
        ).ravel()

        type_metrics.append({
            "clone_type": clone_type,
            "total_pairs": len(group),
            "accuracy (%)": round(
                accuracy_score(y_true_type, y_pred_type) * 100, 2
            ),
            "precision (%)": round(
                precision_score(y_true_type, y_pred_type, zero_division=0) * 100, 2
            ),
            "recall (%)": round(
                recall_score(y_true_type, y_pred_type, zero_division=0) * 100, 2
            ),
            "f1_score (%)": round(
                f1_score(y_true_type, y_pred_type, zero_division=0) * 100, 2
            ),
            "TP": tp_t,
            "FP": fp_t,
            "FN": fn_t,
            "TN": tn_t,
            "avg_similarity": round(
                group["jplag_similarity_score"].mean(), 2
            )
        })
    type_metrics_df = pd.DataFrame(type_metrics)
    
    # ------------------------------------------------------------------
    # EXPORT EXCEL
    # ------------------------------------------------------------------
    os.makedirs(os.path.dirname(output_excel), exist_ok=True)
    with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
        result_df.to_excel(writer, sheet_name="pairs_prediction_log", index=False)
        metrics_df.to_excel(writer, sheet_name="metrics_summary", index=False)
        clone_summary.to_excel(writer, sheet_name="clone_summary", index=False)
        type_metrics_df.to_excel(writer, sheet_name="clone_type_metrics", index=False)

        for ws in writer.sheets.values():
            for col in ws.columns:
                max_len = max(len(str(cell.value or "")) for cell in col)
                ws.column_dimensions[col[0].column_letter].width = min(max_len + 3, 50)

    # ------------------------------------------------------------------
    # CONSOLE REPORT
    # ------------------------------------------------------------------
    print(f"{'  JPLAG EVALUATION FINISHED  ':=^70}")
    print("-" * 70)
    print(f"Matched Pairs      : {matched_pairs:,}")
    print(f"Missing Pairs      : {missing_pairs:,}")
    print(f"Coverage           : {matched_pairs / len(result_df) * 100:.2f}%")
    print("-" * 70)
    print(f"Pairs Evaluated    : {len(result_df):,}")
    print(f"Threshold          : {threshold:.2f}%")
    print(f"Accuracy           : {metrics_df['Accuracy (%)'].values[0]:.2f}%")
    print(f"Precision          : {metrics_df['Precision (%)'].values[0]:.2f}%")
    print(f"Recall             : {metrics_df['Recall (%)'].values[0]:.2f}%")
    print(f"F1 Score           : {metrics_df['F1 Score (%)'].values[0]:.2f}%")
    print(f"Confusion Matrix   : [TP:{tp}, FP:{fp}, FN:{fn}, TN:{tn}]")
    print(f"ROC-AUC            : {round(roc_auc, 2) * 100}")
    print(f"Output             : {output_excel}")
    print("\nPER TYPE METRICS")
    print("-" * 70)
    
    return result_df, type_metrics_df

In [8]:
print(f"{'EVAL PRAKTIKUM':^70}")
result_P, type_klon_P = evaluate_jplag_from_pairs(
    evaluation_excel=r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\output(2)\eval\praktikum\evaluasi_cosine.xlsx",
    jplag_csv=r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_eval_prak\results\results.csv",
    output_excel=r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\output(2)\eval\praktikum\evaluasi_jplag.xlsx",
    threshold=80.0
)

display(result_P.head())
print("=" * 70)
display(type_klon_P.head())

                            EVAL PRAKTIKUM                            
Pairs Evaluation : 4,024
JPlag Results    : 3,161,355
====================  JPLAG EVALUATION FINISHED  =====================
----------------------------------------------------------------------
Matched Pairs      : 4,024
Missing Pairs      : 0
Coverage           : 100.00%
----------------------------------------------------------------------
Pairs Evaluated    : 4,024
Threshold          : 80.00%
Accuracy           : 76.17%
Precision          : 99.62%
Recall             : 52.53%
F1 Score           : 68.79%
Confusion Matrix   : [TP:1057, FP:4, FN:955, TN:2008]
ROC-AUC            : 91.0
Output             : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\output(2)\eval\praktikum\evaluasi_jplag.xlsx

PER TYPE METRICS
----------------------------------------------------------------------


,clone_type,file_1,file_2,jplag_similarity_score,ground_truth,prediction,result
0,type1,MHS001_js02_p01,MHS001_js02_p01_type1,100.00,1,1,CORRECT
1,type2,MHS001_js02_p01,MHS001_js02_p01_type2,100.00,1,1,CORRECT
2,type3,MHS001_js02_p01,MHS001_js02_p01_type3,78.57,1,0,WRONG
3,type4,MHS001_js02_p01,MHS001_js02_p01_type4,0.00,1,0,WRONG
4,type1,MHS001_js02_p02,MHS001_js02_p02_type1,100.00,1,1,CORRECT


,clone_type,total_pairs,accuracy (%),precision (%),recall (%),f1_score (%),TP,FP,FN,TN,avg_similarity
0,negative,2012,99.80,0.0,0.00,0.00,0,4,0,2008,9.14
1,type1,503,100.00,100.0,100.00,100.00,503,0,0,0,100.00
2,type2,503,100.00,100.0,100.00,100.00,503,0,0,0,99.08
3,type3,503,10.14,100.0,10.14,18.41,51,0,452,0,59.88
4,type4,503,0.00,0.0,0.00,0.00,0,0,503,0,14.54


In [9]:
print(f"{'EVAL TUGAS':^70}")
result_T, type_klon_T = evaluate_jplag_from_pairs(
    evaluation_excel=r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\output(2)\eval\tugas\evaluasi_cosine.xlsx",
    jplag_csv=r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\jplag\output_jplag_eval_tgs\results\results.csv",
    output_excel=r"D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\output(2)\eval\tugas\evaluasi_jplag.xlsx",
    threshold=80.0
)
display(result_T.head())
print("=" * 70)
display(type_klon_T.head())

                              EVAL TUGAS                              
Pairs Evaluation : 1,360
JPlag Results    : 360,825
====================  JPLAG EVALUATION FINISHED  =====================
----------------------------------------------------------------------
Matched Pairs      : 1,360
Missing Pairs      : 0
Coverage           : 100.00%
----------------------------------------------------------------------
Pairs Evaluated    : 1,360
Threshold          : 80.00%
Accuracy           : 75.88%
Precision          : 100.00%
Recall             : 51.76%
F1 Score           : 68.22%
Confusion Matrix   : [TP:352, FP:0, FN:328, TN:680]
ROC-AUC            : 97.0
Output             : D:\PUTRI\D4\SEMESTER 8 (4C)\SKRIPSI\code\output(2)\eval\tugas\evaluasi_jplag.xlsx

PER TYPE METRICS
----------------------------------------------------------------------


,clone_type,file_1,file_2,jplag_similarity_score,ground_truth,prediction,result
0,type1,MHS001_js03_tp,MHS001_js03_tp_type1,100.00,1,1,CORRECT
1,type2,MHS001_js03_tp,MHS001_js03_tp_type2,99.73,1,1,CORRECT
2,type3,MHS001_js03_tp,MHS001_js03_tp_type3,66.83,1,0,WRONG
3,type4,MHS001_js03_tp,MHS001_js03_tp_type4,35.78,1,0,WRONG
4,type1,MHS002_js04_tp,MHS002_js04_tp_type1,100.00,1,1,CORRECT


,clone_type,total_pairs,accuracy (%),precision (%),recall (%),f1_score (%),TP,FP,FN,TN,avg_similarity
0,negative,680,100.00,0.0,0.00,0.00,0,0,0,680,1.70
1,type1,170,100.00,100.0,100.00,100.00,170,0,0,0,100.00
2,type2,170,99.41,100.0,99.41,99.71,169,0,1,0,99.13
3,type3,170,7.65,100.0,7.65,14.21,13,0,157,0,61.11
4,type4,170,0.00,0.0,0.00,0.00,0,0,170,0,13.74
